# Validation: Seed Stability

- **Purpose:** Retrain the joint winner with seeds 1 and 2 and compare scores for each cell line and dominant neighbor labels with seed 0.
- **Inputs:** Processed H5ADs; joint winner config and model.
- **Outputs:** Metrics printed in the notebook; artifacts under `models/validation/seed_stability/`.
- **Result:** The mean cell line score stays within 0.674-0.682, and the dominant Pelka label is unchanged for all ten cell lines.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import anndata as ad
import scvi

from src import hpo, line_ranking
from src import hpo_pp_utils as ppu
from src import model_io
from src.utils import load_config

CFG = load_config()
PATHS = CFG["paths"]
HPO_CFG = CFG["hpo"]

adata_tumor = ad.read_h5ad(ROOT / PATHS["processed"]["pp_tumor"])
adata_cl = ad.read_h5ad(ROOT / PATHS["processed"]["pp_cell_line"])
print(f"tumor {adata_tumor.shape}, cell line {adata_cl.shape}")

tumor (108131, 28617), cell line (2162, 28617)


# Joint winner (fixed k=100)

In [2]:
best = pd.read_csv(ROOT / "models" / "hpo" / "best_models_primary_100.csv")
JOINT_RUN = ROOT / best.loc[best["approach"] == "joint", "run_dir"].iloc[0]
joint_cfg = json.loads((JOINT_RUN / "config.json").read_text())
print(JOINT_RUN)

/mnt/bigdisk/markus/CRC-scMFmap/models/hpo/runs/hvg3000/config_10/joint


## Retrain with seeds 1 and 2

In [3]:
torch.set_float32_matmul_precision("high")
combined_3k = model_io.build_combined_adata_for_hvg(
    adata_tumor, adata_cl, joint_cfg["num_hvg"], hvg_dir=JOINT_RUN.parents[1]
)

In [4]:
SEED_RUNS = {0: JOINT_RUN}
for seed in (1, 2):
    cfg_seed = {**joint_cfg, "seed": seed}
    hpo.set_seed(seed)
    seed_dir = ROOT / "models" / "validation" / "seed_stability" / f"seed_{seed}"
    model_path = seed_dir / "joint_scanvi"
    if model_path.exists():
        print(f"[load] seed {seed}: {model_path}")
        model = scvi.model.SCANVI.load(str(model_path), adata=combined_3k)
    else:
        print(f"[train] seed {seed} -> {seed_dir}")
        model = hpo.train_joint_scanvi(combined_3k, cfg_seed, seed_dir)
    if hpo.is_run_done(seed_dir):
        print(f"[skip] seed {seed} evaluation already done")
    else:
        hpo.evaluate(
            model, combined_3k, cfg_seed, seed_dir, "joint",
            scib_n_cells=HPO_CFG["scib_n_cells"],
        )
        hpo.mark_run_done(seed_dir)
    SEED_RUNS[seed] = seed_dir

Seed set to 1


[train] seed 1 -> /mnt/bigdisk/markus/CRC-scMFmap/models/validation/seed_stability/seed_1


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/bigdisk/markus/miniconda3/envs/scvi/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/mnt/bigdisk/markus/miniconda3/envs/scvi/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve p

Training:   0%|          | 0/400 [00:00<?, ?it/s]

INFO     File                                                                                                      
         /mnt/bigdisk/markus/CRC-scMFmap/notebooks/curr/scvi_log/2026-08-31_18-36-03_elbo_validation/epoch=147-step
         =57424-elbo_validation=1406.9874267578125/model.pt already downloaded                                     
Monitored metric elbo_validation did not improve in the last 15 records. Best score: 1406.987. Signaling Trainer to stop.
INFO     Training for 400 epochs.                                                                                  


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/bigdisk/markus/miniconda3/envs/scvi/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/mnt/bigdisk/markus/miniconda3/envs/scvi/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve p

... (7 outputs left out)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/bigdisk/markus/miniconda3/envs/scvi/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/mnt/bigdisk/markus/miniconda3/envs/scvi/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve p

Training:   0%|          | 0/400 [00:00<?, ?it/s]

INFO     File                                                                                                      
         /mnt/bigdisk/markus/CRC-scMFmap/notebooks/curr/scvi_log/2026-08-31_19-20-31_elbo_validation/epoch=42-step=
         16684-elbo_validation=1406.183349609375/model.pt already downloaded                                       
Monitored metric elbo_validation did not improve in the last 15 records. Best score: 1406.216. Signaling Trainer to stop.


/mnt/bigdisk/markus/miniconda3/envs/scvi/lib/python3.11/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)


## Line score spread at k=100
Per-line line score (geometric mean of mixing, specificity, within-line
purity) per seed, with mean/sd across seeds.

In [5]:
from sklearn.neighbors import NearestNeighbors


def seed_line_metrics(seed):
    cells = pd.read_csv(SEED_RUNS[seed] / "cell_line_mixing.csv")
    line = line_ranking.line_metrics_from_cells(
        cells, "mixing_k100", "entropy_k100", "coverage_k100", "line_purity_k100"
    )
    return line.set_index("cell_line")


line_by_seed = {seed: seed_line_metrics(seed) for seed in SEED_RUNS}

In [6]:
scores = pd.DataFrame({seed: lm["line_score"] for seed, lm in line_by_seed.items()})
scores.columns = [f"seed_{s}" for s in scores.columns]
scores["mean"] = scores.mean(axis=1)
scores["sd"] = scores.std(axis=1)
scores.round(4)

,seed_0,seed_1,seed_2,mean,sd
cell_line,,,,,
NCIH747,0.5159,0.5279,0.5729,0.5389,0.0245
LS1034,0.7719,0.7842,0.7706,0.7756,0.0061
RCM1,0.6536,0.5913,0.6616,0.6355,0.0314
CCK81,0.6314,0.6720,0.6665,0.6566,0.0180
SNUC4,0.7122,0.7127,0.7195,0.7148,0.0033
CL34,0.6493,0.6605,0.6509,0.6536,0.0049
LS180,0.7547,0.6888,0.7283,0.7239,0.0271
OUMS23,0.6438,0.7101,0.6310,0.6616,0.0347
HCC56,0.7049,0.6786,0.6992,0.6942,0.0113


In [7]:
pd.DataFrame({
    "seed": list(line_by_seed),
    "mean_line_score": [line_by_seed[s]["line_score"].mean() for s in line_by_seed],
    "mean_mixing": [line_by_seed[s]["mixing"].mean() for s in line_by_seed],
    "mean_specificity": [line_by_seed[s]["specificity"].mean() for s in line_by_seed],
    "mean_line_purity": [line_by_seed[s]["line_purity"].mean() for s in line_by_seed],
}).round(4)

,seed,mean_line_score,mean_mixing,mean_specificity,mean_line_purity
0,0,0.6791,0.6923,0.9496,0.5224
1,1,0.6739,0.6775,0.9450,0.5182
2,2,0.6824,0.6783,0.9419,0.5371


# Dominant tumor label stability
Per cell line: the mode over cells of the dominant Pelka label among the
k=100 nearest tumor neighbors (latent from each seed's saved model), plus
the fraction of the line's cells sharing that dominant label. Then stability
across seeds: how often the dominant label coincides and the mean pairwise
Jaccard overlap of the top-3 label sets.

In [8]:
K = 100


def seed_dominant_labels(seed):
    model = scvi.model.SCANVI.load(str(SEED_RUNS[seed] / "joint_scanvi"), adata=combined_3k)
    latent = model.get_latent_representation(combined_3k)
    obs = combined_3k.obs
    tumor_mask = (obs["dataset"] == "tumor").to_numpy()
    cl_idx = np.where(~tumor_mask)[0]
    tumor_idx = np.where(tumor_mask)[0]

    nn = NearestNeighbors(n_neighbors=K, n_jobs=1)
    nn.fit(latent[tumor_idx])
    _, ind = nn.kneighbors(latent[cl_idx])
    labels = obs["label"].to_numpy()[tumor_idx][ind]
    lines = obs["cell_line"].to_numpy()[cl_idx]

    records = []
    for row, line in enumerate(lines):
        lab, count = np.unique(labels[row], return_counts=True)
        if len(lab) == 0:
            records.append({"cell_line": line, "top1": None, "top2": None,
                            "top3": None, "top1_frac": np.nan})
            continue
        order = np.argsort(-count)
        records.append({
            "cell_line": line,
            "top1": lab[order[0]],
            "top2": lab[order[1]] if len(order) > 1 else None,
            "top3": lab[order[2]] if len(order) > 2 else None,
            "top1_frac": count[order[0]] / K,
        })
    del model
    return pd.DataFrame(records)


dom_by_seed = {seed: seed_dominant_labels(seed) for seed in SEED_RUNS}

INFO     File /mnt/bigdisk/markus/CRC-scMFmap/models/hpo/runs/hvg3000/config_10/joint/joint_scanvi/model.pt already
         downloaded                                                                                                
INFO     File /mnt/bigdisk/markus/CRC-scMFmap/models/validation/seed_stability/seed_1/joint_scanvi/model.pt already
         downloaded                                                                                                
INFO     File /mnt/bigdisk/markus/CRC-scMFmap/models/validation/seed_stability/seed_2/joint_scanvi/model.pt already
         downloaded                                                                                                


In [9]:
def modal(series):
    m = series.dropna().mode()
    return m.iloc[0] if len(m) else None


def line_label_summary(seed):
    grouped = dom_by_seed[seed].groupby("cell_line")
    return pd.DataFrame({
        "dominant_label": grouped["top1"].agg(modal),
        "dominant_frac": grouped["top1"].agg(
            lambda s: float((s.dropna() == modal(s)).mean())),
        "top2": grouped["top2"].agg(modal),
        "top3": grouped["top3"].agg(modal),
    })


label_by_seed = {seed: line_label_summary(seed) for seed in SEED_RUNS}
pd.concat(label_by_seed, names=["seed", "cell_line"]).round(4)

dominant_label  dominant_frac  \
seed cell_line                                                    
0    CCK81      Tumor cE03 (Stem/TA-like prolif)         0.9449   
     CL34       Tumor cE03 (Stem/TA-like prolif)         0.9713   
     HCC56      Tumor cE03 (Stem/TA-like prolif)         0.5980   
     HT55       Tumor cE03 (Stem/TA-like prolif)         0.9289   
     LS1034            Tumor cE01 (Stem/TA-like)         0.5229   
     LS180      Tumor cE03 (Stem/TA-like prolif)         0.7928   
     NCIH747    Tumor cE03 (Stem/TA-like prolif)         0.9000   
     OUMS23     Tumor cE03 (Stem/TA-like prolif)         0.8879   
     RCM1       Tumor cE03 (Stem/TA-like prolif)         0.8244   
     SNUC4      Tumor cE03 (Stem/TA-like prolif)         0.9363   
1    CCK81      Tumor cE03 (Stem/TA-like prolif)         0.9685   
     CL34       Tumor cE03 (Stem/TA-like prolif)         0.9713   
     HCC56      Tumor cE03 (Stem/TA-like prolif)         0.6765   
     HT55       Tumor cE03 (Stem/TA-like prolif)         0.9242   
     LS1034            Tumor cE01 (Stem/TA-like)         0.5425   
     LS180      Tumor cE03 (Stem/TA-like prolif)         0.7293   
     NCIH747    Tumor cE03 (Stem/TA-like prolif)         0.9000   
     OUMS23     Tumor cE03 (Stem/TA-like prolif)         0.8131   
     RCM1       Tumor cE03 (Stem/TA-like prolif)         0.8168   
     SNUC4      Tumor cE03 (Stem/TA-like prolif)         0.9384   
2    CCK81      Tumor cE03 (Stem/TA-like prolif)         0.9370   
     CL34       Tumor cE03 (Stem/TA-like prolif)         0.9737   
     HCC56      Tumor cE03 (Stem/TA-like prolif)         0.7255   
     HT55       Tumor cE03 (Stem/TA-like prolif)         0.9289   
     LS1034            Tumor cE01 (Stem/TA-like)         0.5294   
     LS180      Tumor cE03 (Stem/TA-like prolif)         0.8094   
     NCIH747    Tumor cE03 (Stem/TA-like prolif)         0.8875   
     OUMS23     Tumor cE03 (Stem/TA-like prolif)         0.7757   
     RCM1       Tumor cE03 (Stem/TA-like prolif)         0.8092   
     SNUC4      Tumor cE03 (Stem/TA-like prolif)         0.9384   

                                                     top2  \
seed cell_line                                              
0    CCK81                      Tumor cE01 (Stem/TA-like)   
     CL34                       Tumor cE01 (Stem/TA-like)   
     HCC56               Tumor cE03 (Stem/TA-like prolif)   
     HT55                       Tumor cE01 (Stem/TA-like)   
     LS1034                     Tumor cE01 (Stem/TA-like)   
     LS180      Tumor cE02 (Stem/TA-like/Immature Goblet)   
     NCIH747                    Tumor cE01 (Stem/TA-like)   
     OUMS23                     Tumor cE01 (Stem/TA-like)   
     RCM1                Tumor cE03 (Stem/TA-like prolif)   
     SNUC4                      Tumor cE01 (Stem/TA-like)   
1    CCK81                      Tumor cE01 (Stem/TA-like)   
     CL34                       Tumor cE01 (Stem/TA-like)   
     HCC56                      Tumor cE01 (Stem/TA-like)   
     HT55                       Tumor cE01 (Stem/TA-like)   
     LS1034                     Tumor cE01 (Stem/TA-like)   
     LS180      Tumor cE02 (Stem/TA-like/Immature Goblet)   
     NCIH747                    Tumor cE01 (Stem/TA-like)   
     OUMS23                     Tumor cE01 (Stem/TA-like)   
     RCM1                       Tumor cE01 (Stem/TA-like)   
     SNUC4                      Tumor cE01 (Stem/TA-like)   
2    CCK81                      Tumor cE01 (Stem/TA-like)   
     CL34                       Tumor cE01 (Stem/TA-like)   
     HCC56                      Tumor cE01 (Stem/TA-like)   
     HT55                       Tumor cE01 (Stem/TA-like)   
     LS1034                     Tumor cE01 (Stem/TA-like)   
     LS180      Tumor cE02 (Stem/TA-like/Immature Goblet)   
     NCIH747                    Tumor cE01 (Stem/TA-like)   
     OUMS23                     Tumor cE01 (Stem/TA-like)   
     RCM1                       Tumor cE01 (Stem/TA-like)  

In [12]:
def top_set(row):
    return {v for v in (row["dominant_label"], row["top2"], row["top3"]) if pd.notna(v)}


stability = []
for line in scores.index:
    doms = [label_by_seed[s].loc[line, "dominant_label"] for s in SEED_RUNS]
    tops = [top_set(label_by_seed[s].loc[line]) for s in SEED_RUNS]
    jaccards = [
        1.0 if not (tops[i] | tops[j])
        else len(tops[i] & tops[j]) / len(tops[i] | tops[j])
        for i, j in ((0, 1), (0, 2), (1, 2))
    ]
    valid_doms = [d for d in doms if pd.notna(d)]
    modal_label = pd.Series(valid_doms).mode().iloc[0] if valid_doms else None
    stability.append({
        "cell_line": line,
        "dominant_labels": "/".join(sorted({d for d in doms if pd.notna(d)})) or "none",
        "top1_agreement": f"{sum(d == modal_label for d in doms)}/{len(doms)}",
        "mean_top3_jaccard": float(np.mean(jaccards)),
    })
pd.DataFrame(stability)

,cell_line,dominant_labels,top1_agreement,mean_top3_jaccard
0,NCIH747,Tumor cE03 (Stem/TA-like prolif),3/3,0.500000
1,LS1034,Tumor cE01 (Stem/TA-like),3/3,1.000000
2,RCM1,Tumor cE03 (Stem/TA-like prolif),3/3,0.500000
3,CCK81,Tumor cE03 (Stem/TA-like prolif),3/3,0.666667
4,SNUC4,Tumor cE03 (Stem/TA-like prolif),3/3,1.000000
5,CL34,Tumor cE03 (Stem/TA-like prolif),3/3,0.777778
6,LS180,Tumor cE03 (Stem/TA-like prolif),3/3,0.777778
7,OUMS23,Tumor cE03 (Stem/TA-like prolif),3/3,1.000000
8,HCC56,Tumor cE03 (Stem/TA-like prolif),3/3,0.777778
9,HT55,Tumor cE03 (Stem/TA-like prolif),3/3,1.000000
